# Week 6 Homework

In [ ]:
DEBUG_MODE    = False # True
# DEBUG_MODE_SELECT = True

In [ ]:
# 0.0 - SETUP
# 0.1 - DATA PREP AND MATH LIBRARIES
import pandas as pd
import numpy as np

# PLOTTING
import seaborn as sns
from matplotlib                 import pyplot as plt
%matplotlib inline                     

# SPLITTING DATASET
from sklearn.model_selection    import train_test_split        # 

# DECISION TREES
from sklearn.feature_extraction import DictVectorizer          # 
from sklearn.metrics            import roc_auc_score           # USED WITH CLASSIFICATION
from sklearn.metrics            import mean_squared_error      # USED WITH REGRESSION
from sklearn.tree               import DecisionTreeClassifier  # 
from sklearn.tree               import DecisionTreeRegressor   # 
from sklearn.ensemble           import RandomForestRegressor   #
from sklearn.tree               import export_text             # 


# ENSEMBLE LEARNING
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb


In [ ]:
# 0.2 - GET DATA

data_url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df       = pd.read_csv(data_url)

# STANDARDIZE ALL COLUMN NAMES
df.columns = df.columns.str.lower()


In [ ]:
# 0.2 - EDA
# CHECK FOR MISSING VALUES
if DEBUG_MODE == True:
    print(f"df.count()\r\n===============================\r\n{df.count()}\r\n")

# CREATE A LIST OF COLUMNS CONTAINING AT LEAST ONE MISSING VALUE
missing_columns = df.columns[df.isna().any()].tolist()

# DISPLAY LIST OF ALL COLUMNS MISSING VALUES
if DEBUG_MODE == True:
    print(f"missing_columns (a/k/a columns missing values)\r\n===============================\r\n{missing_columns}")


In [ ]:
# CREATE A LIST OF CATEGORICAL COLUMNS
categorical_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()  

# REMOVE WHITESPACES AND SHOW FREQUENCY OF UNIQUE CATEGORICAL VALUES BY COLUMN
for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')
    
    if DEBUG_MODE == True:
        print(df[c].value_counts())
    

In [ ]:
# CREATE A LIST OF NUMERIC COLUMNS
numerical_columns   = df.select_dtypes(include="number").columns.tolist()


In [ ]:
# CREATE A LIST OF NUMERIC COLUMNS
numerical_columns   = df.select_dtypes(include="number").columns.tolist()

# CREATE A LIST OF COLUMNS CONTAINING AT LEAST ONE MISSING VALUE
# missing_columns = df.columns[df.isna().any()].tolist() 

# FILL MISSING NUMERIC VALUES WITH ZEROS
for n in missing_columns:
    df[n] = pd.to_numeric(
                df[n], 
                errors='coerce'
            )
    df[n] = df[n].fillna(0)

# DESCRIBE STATISTICAL ATTRIBUTES OF EACH NUMERICAL COLUMN
if DEBUG_MODE == True:
    for n in numerical_columns:
        print(df[n].describe)

In [ ]:
# PREVIEW DATASET
if DEBUG_MODE == True:
    df.head(25)

In [ ]:
# SPLIT THE DATASET
df_full_train, df_test  = train_test_split(
                            df, 
                            test_size    = 0.2, 
                            random_state = 1
                          )
df_train, df_val        = train_test_split(
                            df_full_train, 
                            test_size    = 0.25, 
                            random_state = 1
                          )

# RESET THE INDEX VALUES FOR EACH SPLIT
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(  drop=True)
df_test  = df_test.reset_index( drop=True)

In [ ]:
if DEBUG_MODE == True:
    df_train.head

In [ ]:
# DEFINE TARGET MATRIX
target_column = 'fuel_efficiency_mpg'

# - CREATE TARGET VECTOR
y_train = (df_train[target_column]) # == 'default').astype('int').values
y_val   = (df_val[target_column])   # == 'default').astype('int').values
y_test  = (df_test[target_column])  # == 'default').astype('int').values

# - DROP TARGET FROM THE FEATURE MATRIX
for df in (df_train, df_val, df_test):
    df.drop(columns=[target_column], inplace=True)

In [ ]:
# DEFINE FEATURE MATRIX
# - CONVERT 'pandas' DATAFRAME TO A DICTIONARY WITH ONE DICTIONARY ENTRY PER ROW
train_dicts = df_train.fillna(0).to_dict(orient='records')
val_dicts   = df_val.fillna(0).to_dict(orient='records')

# - CREATE VECTORIZER
dv          = DictVectorizer(sparse=False)

# - USE VECTORIZER TO TURN FEATURE DICTIONARY INTO TO A MATRIX
X_train     = dv.fit_transform(train_dicts)
X_val       = dv.transform(val_dicts)

In [ ]:
# DEBUG - PREVIEW 'X_train'
if DEBUG_MODE == True:
    np.set_printoptions(suppress=True)
    print(X_train[:5].round(3))
    np.set_printoptions(suppress=False)

# Question 1

Let's train a decision tree regressor to predict the fuel_efficiency_mpg variable.

    Train a model with max_depth=1.

Which original feature is used for splitting the data? If you use DictVectorizer, map a one-hot feature such as origin=USA back to its original column name.

    'vehicle_weight'
    'model_year'
    'origin'
    'fuel_type'


In [ ]:
# Q1
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor()
dt.fit(X_train, y_train)

In [ ]:
# Q1
# DEBUG - PREVIEW FEATURE NAMES TO CONFIRM  DROPPED TARGET COLUMN AND ONE-HOT ENCODING
if DEBUG_MODE == True:
    print(dv.get_feature_names_out())


In [ ]:
# Q1 - PART 1
#   NOTE: THESE ARE THE IMPORTANT METHODS
#     from sklearn.tree    import DecisionTreeRegressor
#     from sklearn.metrics import mean_squared_error

#   NOTE: CODE REQUIRES A REGRESSION METHOD, NOT A CLASSIFIER METHOD
#    CLASSIFICATION PREDICTIONS USE 'predict_proba()'
#    REGRESSION PREDICTIONS USE 'predict()'

# SET FIRST MODEL TUNING PARAMETER
depth            = 1

# TRAIN THE DECISION TREE
dt               = DecisionTreeRegressor(
                    max_depth=depth
                   )
dt.fit(X_train, y_train)
print(f"Q1 - PART 1 ANSWER: \r\n\tModel trained with 'max_depth':\t{depth}\r\n")


# Q1 - PART 2
#   DETERMINE WHICH FEATURE WAS USED FOR SPLITTING THE DATA
#   - GET A LIST OF FEATURE NAMES
feature_names = dv.get_feature_names_out()
#   - PULL INDEX OF FIRST FEATURE NAME USED IN TRAINING, THEN MAP THE INDEX POSITION TO A FEATURE NAME
split_feature = feature_names[dt.tree_.feature[0]]
print(f"Q1 - PART 2 ANSWER: \r\n\tsplit_feature: \t\t\t'{split_feature}'\r\n")

# Q1 - PART 3
#   RESTORE ORIGINAL FEATURE NAME COLUMN FOR ONE ONE-HOT ENCODED VARIABLE
split_feature    = dv.get_feature_names_out()[dt.tree_.feature[0]]
original_feature = split_feature.split("=")[0]

print(f"Q1 - PART 3 ANSWER: \r\n\tOriginal feature name:\t\t'{original_feature}'")


# Question 2

Train a random forest regressor with these parameters:

    n_estimators=10
    random_state=1
    n_jobs=-1 (optional - to make training faster)

What's the RMSE of this model on the validation data?

    0.1837
    1.837
    18.37
    183.7


In [ ]:
# Q2 - TEST WITH PREDETERMINED PARAMETERS
#   NOTE: THESE ARE THE IMPORTANT METHODS
#     from sklearn.ensemble import RandomForestRegressor
#     from sklearn.metrics  import mean_squared_error

# SET RANDOM FOREST TUNING PARAMETERS EXPLICITLY
rf    = RandomForestRegressor(
            n_estimators = 10,
            random_state = 1,
            n_jobs       = -1
        )
# TRAIN RANDOM FOREST WITH TUNING PARAMETERS
rf.fit(X_train, y_train)

# MAKE PREDICTION
y_val_pred = rf.predict(X_val)

# EVALUATE PERFORMANCE
rmse = mean_squared_error(y_val, y_val_pred) ** 0.5

print(f"Q2 ANSWER:\r\n\trmse:\t{rmse}")


# Question 3

Now let's experiment with the n_estimators parameter

    Try these values of this parameter: [10, 50, 100, 150].
    Set random_state to 1.
    Evaluate the model on the validation dataset.

Which value gives the lowest validation RMSE? Consider 3 decimal places for calculating the answer.

    10
    50
    100
    150


In [ ]:
# Q3 - ITERATE THROUGH POSSIBLE n_estimators TUNING PARAMETERS

n_estimators = [10, 50, 100, 150]
for estimator in n_estimators:

    rf    = RandomForestRegressor(
            n_estimators = estimator,
            random_state = 1,
            n_jobs       = -1
        )
    
    # TRAIN RANDOM FOREST WITH TUNING PARAMETERS
    rf.fit(X_train, y_train)

    # MAKE PREDICTION
    y_val_pred = rf.predict(X_val)

    # EVALUATE PERFORMANCE
    rmse = mean_squared_error(y_val, y_val_pred) ** 0.5
    print(f"n_estimators={estimator}: RMSE={rmse:.3f}")

print("A RMSE of zero is ideal")

# Question 4

Let's select the best max_depth:

    Try different values of max_depth: [10, 15, 20, 25]
    For each of these values,
        try different values of n_estimators from [10, 50, 100, 150]
        calculate the mean RMSE
    Fix the random seed: random_state=1

What's the best max_depth, using the mean RMSE?

    10
    15
    20
    25


In [ ]:
# Q4 - ITERATE THROUGH POSSIBLE n_estimators TUNING PARAMETERS

# MODEL TUNING PARAMETERS
max_depth     = [10, 15, 20, 25]
n_estimators  = [10, 50, 100, 150]
random_num_seed = 1

# LIST OF ITERATIVE TESTING RESULTS
results = []

# CALCULATE RMSE FOR EACH COMBINATION OF PARAMETER VALUES
for depth in max_depth:
    for estimator in n_estimators:
        rf    = RandomForestRegressor(
                    max_depth    = depth,
                    n_estimators = estimator,
                    random_state = random_num_seed,
                    n_jobs       = -1
                )
    
        # TRAIN RANDOM FOREST WITH TUNING PARAMETERS
        rf.fit(X_train, y_train)

        # MAKE PREDICTION
        y_val_pred = rf.predict(X_val)

        # EVALUATE PERFORMANCE
        rmse       = mean_squared_error(y_val, y_val_pred) ** 0.5

        results.append({
            "estimator": estimator,
            "depth": depth,
            "rmse": rmse
        })

df_results = pd.DataFrame(results)

print(df_results.round({"rmse": 3}))

best_row = df_results.loc[df_results["rmse"].idxmin()]
print("\nLowest-RMSE row:")
print(f"\r\n{best_row.to_string(float_format=lambda value: f'{value:.3f}')}")

print("A RMSE of zero is ideal")

print(f"\r\nQ4 ANSWER\r\n\tBest max_depth: {best_row['depth']}")


# Question 5

We can extract feature importance information from tree-based models.

At each step of the decision tree learning algorithm, it finds the best split. When doing it, we can calculate "gain" - the reduction in impurity before and after the split. This gain is quite useful in understanding what are the important features for tree-based models.

In Scikit-Learn, tree-based models contain this information in the feature_importances_ field.

For this homework question, we'll find the most important feature:

    Train the model with these parameters:
        n_estimators=10,
        max_depth=20,
        random_state=1,
        n_jobs=-1 (optional)
    Get the feature importance information from this model

What's the most important feature (among these 4)?

    vehicle_weight
    horsepower
    acceleration
    engine_displacement



In [ ]:
# Q5
estimator       = 10
depth           = 20
random_num_seed = 1

rf              = RandomForestRegressor(
                    n_estimators = estimator,
                    max_depth    = depth,
                    
                    random_state = random_num_seed,
                    n_jobs       = -1
                  )
    
# TRAIN RANDOM FOREST WITH TUNING PARAMETERS
rf.fit(X_train, y_train)

# MAKE PREDICTION
y_val_pred = rf.predict(X_val)

# CREATE A DATAFRAME OF FEATURE IMPORTANCE
feature_importance = pd.DataFrame(
                        {
                            "feature": dv.get_feature_names_out(),
                            "importance": rf.feature_importances_
                         }
                      ).sort_values(
                        "importance", 
                        ascending = False
                      ).reset_index(drop=True)

print(feature_importance)

print("Q5 ANSWER:\r\n\t'vehicle_weight' is the most important feature in this list of features: 'vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement'")





# Question 6

Now let's train an XGBoost model! For this question, we'll tune the eta parameter:

    Install XGBoost
    Create DMatrix for train and validation
    Create a watchlist
    Train a model with these parameters for 100 rounds:

xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}

Now change eta from 0.3 to 0.1.

Which eta leads to the best RMSE score on the validation dataset?

    0.3
    0.1
    Both give equal value


In [ ]:
# Q6 PART 2 - CREATE A DMatrix FOR TRAIN AND VALIDATION
# import xgboost as xgb

dm_train        = xgb.DMatrix(
                    X_train, 
                    label=y_train
                  )
dm_val          = xgb.DMatrix(
                    X_val, 
                    label = y_val
                  )

# watchlist     = [(dm_train, "train"), (dm_val, "val")]
eta_list        = [0.3, 0.1]

results         = []

for eta_val in eta_list:
    xgb_params  = {
                    "eta":              eta_val,
                    "max_depth":        6,
                    "min_child_weight": 1,
                    "objective":        "reg:squarederror",
                    "nthread":          8,
                    "seed":             1,
                    "verbosity":        1,
                  }
                
    model       = xgb.train(
                    xgb_params,
                    dm_train,
                    num_boost_round     = 100,
                    # evals               = watchlist
                  )

    # EVALUATE PERFORMANCE
    rmse        = mean_squared_error(y_val, y_val_pred) ** 0.5

    results.append(
                    {
                        "eta":          eta_val,
                        "rmse":         rmse
                    }
                  )

df_results = pd.DataFrame(results)

print(df_results.round({"rmse": 3}))

print("\r\nQ6 ANSWER - PART 2 AND 3\r\n\tBoth 'eta' values return the same RMSE")